# Notebook 05: Specialized Production, Streaming & Probabilistic Trees
## Google YDF, NGBoost Uncertainty & Real-Time Hoeffding Trees

---

### 1. Executive Intuition & Conceptual Roadmap

#### The Mental Model: Beyond Static Batch Training
In traditional machine learning, models are trained on static historical tables. But modern e-commerce systems face three extreme production demands that standard GBDTs cannot handle:
1. **Real-Time Streaming (Kafka / Flink)**: Millions of transactions stream in 24/7. Re-training an XGBoost model every hour causes system downtime and high compute costs. We need **online trees** that learn from one sample at a time without retaining historical data (**Hoeffding Trees**).
2. **Risk & Uncertainty Quantification**: Predicting a single department is insufficient for high-value orders (e.g. luxury watches vs. mobile phones). The business requires full predictive probability distributions and confidence intervals (**NGBoost - Natural Gradient Boosting**).
3. **Microsecond Latency Serving**: When evaluating bids in real-time ad auctions or search rankers, Python runtime overhead (5–20 milliseconds) is unacceptable. We need pure C++ compiled decision forests running in **under 10 microseconds** (**Google Yggdrasil Decision Forests - YDF**).
4. **Unsupervised Bot & Fraud Isolation**: Detecting fraudulent carding attacks and bots before they execute (**Isolation Forest**).

```mermaid
graph LR
    Kafka[Real-Time Event Stream] --> Anomaly[Isolation Forest: Path Length Scoring]
    Anomaly -->|Normal| OnlineTree[Hoeffding Tree: Single-Pass Split Update]
    Anomaly -->|Outlier / Fraud| Block[Fraud Block Action]
    OnlineTree --> ServingEngine[Google YDF: Sub-Microsecond C++ Serving]
```

--- 

### 2. Deep Mathematical Derivations

#### A. Hoeffding Bound for Streaming Online Trees
How can a decision tree decide to create a split without scanning the full historical dataset? It relies on **Wassily Hoeffding's 1963 Inequality**:

##### 1. Mathematical Theorem
Let $r_1, r_2, \dots, r_n$ be independent random variables bounded in range $R$ (e.g. for Gini impurity, $R = 1.0$). Let $\bar{r} = \frac{1}{n} \sum r_i$ be their empirical sample mean, and $\mathbb{E}[r]$ be the true underlying mean. Then for any $\epsilon > 0$:
$$\mathbb{P}\left( |\bar{r} - \mathbb{E}[r]| \ge \epsilon \right) \le 2 \exp\left( -\frac{2 n \epsilon^2}{R^2} \right)$$

##### 2. Solving for the Decision Threshold $\epsilon$
Set the failure probability threshold to $\delta$ (e.g. $\delta = 10^{-5}$):
$$\delta = 2 \exp\left( -\frac{2 n \epsilon^2}{R^2} \right) \implies \ln(\delta / 2) = -\frac{2 n \epsilon^2}{R^2}$$
$$\mathbf{\epsilon = \sqrt{\frac{R^2 \ln(2 / \delta)}{2n}}}$$

##### 3. The Online Splitting Guarantee
At an active leaf node in a streaming tree:
* Let $X_a$ be the attribute that yields the highest empirical Information Gain $\bar{G}(X_a)$.
* Let $X_b$ be the second-best attribute with gain $\bar{G}(X_b)$.
* Difference: $\Delta \bar{G} = \bar{G}(X_a) - \bar{G}(X_b)$.

**Mathematical Decision Rule**:
If $\mathbf{\Delta \bar{G} > \epsilon}$, the Hoeffding Bound mathematically guarantees (with confidence $1 - \delta$) that $X_a$ is the **true globally optimal split**, even though the tree has only seen $n$ streaming samples! The leaf splits immediately, and historical data can be safely discarded.

#### B. NGBoost: Natural Gradients & Riemannian Geometry

##### 1. Why Standard Euclidean Gradients Fail
Standard gradient boosting updates predictions along the Euclidean negative gradient: $-\nabla_\theta L$. However, probability distributions live on a curved **Riemannian statistical manifold**, where distance is measured by the Kullback-Leibler (KL) divergence, not Euclidean distance:
$$D_{\text{KL}}(P_\theta \parallel P_{\theta + d\theta}) \approx \frac{1}{2} d\theta^T \mathcal{I}(\theta) d\theta$$
where $\mathcal{I}(\theta)$ is the **Fisher Information Matrix (FIM)**:
$$\mathcal{I}(\theta) = \mathbb{E}_{y \sim P_\theta} \left[ \nabla_\theta \ln P(y \mid \theta) \nabla_\theta \ln P(y \mid \theta)^T \right]$$

##### 2. The Natural Gradient Direction
To find the step $d\theta$ that minimizes loss subject to a small constant KL divergence step $\frac{1}{2} d\theta^T \mathcal{I}(\theta) d\theta = \epsilon^2$, we solve the Lagrangian:
$$\max_{d\theta} -\nabla_\theta L^T d\theta \quad \text{s.t.} \quad d\theta^T \mathcal{I}(\theta) d\theta = \epsilon^2$$
The analytical solution is the **Natural Gradient**:
$$\mathbf{\widetilde{\nabla}_\theta L = \mathcal{I}(\theta)^{-1} \nabla_\theta L}$$
NGBoost fits individual trees to the natural gradients of all distribution parameters simultaneously (e.g. mean $\mu$ and variance $\sigma^2$), producing **invariant, well-calibrated prediction intervals**.

#### C. Isolation Forest: Unsupervised Path Length Math
Isolation Forest is built on an intuitive mathematical principle:
> **Anomalies are few and different; therefore, they isolate much closer to the tree root than normal points.**

##### 1. Average Path Length of Unsuccessful Search in BST
Given a dataset of $n$ instances, the average path length of an unsuccessful search in a Binary Search Tree (equivalent to random recursive partitioning) is:
$$c(n) = 2 \ln(n - 1) + 2\gamma - \frac{2(n - 1)}{n}$$
where $\gamma \approx 0.5772156649$ is the Euler-Mascheroni constant.

##### 2. Anomaly Scoring Function
Let $h(x)$ be the depth (number of edges) traversed to isolate sample $x$, and $\mathbb{E}[h(x)]$ be its average path length across an ensemble of random isolation trees. The anomaly score is:
$$\mathbf{s(x, n) = 2^{-\frac{\mathbb{E}[h(x)]}{c(n)}}}$$

**Boundary Analysis**:
* If $\mathbb{E}[h(x)] \to 0$ (isolated near root in 1–2 splits): $s \to 2^0 = \mathbf{1.0}$ (Definite Fraud / Outlier!).
* If $\mathbb{E}[h(x)] \to n - 1$ (isolated at extreme leaf depth): $s \to 2^{-\infty} = \mathbf{0.0}$ (Normal instance).
* If $\mathbb{E}[h(x)] \to c(n)$ (isolated at average depth): $s \to 2^{-1} = \mathbf{0.5}$ (Indistinguishable from average data).

In [ ]:
import sys, os
cur_dir = os.path.abspath(os.getcwd())
trees_dir = os.path.abspath(os.path.join(cur_dir, '..')) if os.path.basename(cur_dir) == 'notebooks' else cur_dir
repo_root = os.path.abspath(os.path.join(trees_dir, '..'))
for p in [trees_dir, repo_root]:
    if p not in sys.path: sys.path.insert(0, p)
# Setup environment and utilities
import sys, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

if root_dir not in sys.path:

from trees.utils import (
    print_hardware_summary, load_dataset, prepare_features,
    evaluate_multiclass_model, plot_confusion_matrix_20, plot_metrics_comparison,
    DEPARTMENTS_EN
)

print_hardware_summary()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

# Load 30,000 samples
df = load_dataset(sample_rows=30_000)
X_raw, y, cat_cols, num_cols = prepare_features(df)

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_encoded = X_raw.copy()
X_encoded[cat_cols] = encoder.fit_transform(X_raw[cat_cols])

# 70/10/20 Stratified Split
X_train, X_temp, y_train, y_temp = train_test_split(X_encoded, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.6667, random_state=42, stratify=y_temp)

print(f"Train instances: {len(X_train):,} | Test instances: {len(X_test):,}")

### 3. Model 1: Google YDF (Yggdrasil Decision Forests)
We train Google's compiled C++ engine, demonstrating microsecond serving latency.

In [ ]:
all_results = []

try:
    import ydf
    
    train_df = X_raw.iloc[X_train.index].copy()
    train_df['target'] = y_train
    test_df = X_raw.iloc[X_test.index].copy()
    test_df['target'] = y_test
    
    t0 = time.time()
    ydf_model = ydf.GradientBoostedTreesLearner(
        label='target',
        num_trees=100,
        shrinkage=0.1
    ).train(train_df)
    ydf_time = time.time() - t0
    
    ydf_prob = ydf_model.predict(test_df)
    ydf_pred = np.argmax(ydf_prob, axis=1)
    ydf_metrics = evaluate_multiclass_model("Google YDF (Gradient Boosted Trees)", y_test, ydf_pred, ydf_prob, ydf_time)
    all_results.append(ydf_metrics)
    print("Google YDF Evaluation:", ydf_metrics)
except ImportError:
    print("ydf not installed; skipping Google YDF benchmark.")

### 4. Model 2: Online Streaming Learning with Hoeffding Trees (`river`)
We simulate real-time Kafka streaming transactions arriving one-by-one, observing the Hoeffding Tree learn continuously on the fly.

In [ ]:
try:
    from river import tree, metrics
    
    ht_model = tree.HoeffdingTreeClassifier(grace_period=100, delta=1e-5)
    stream_acc = metrics.Accuracy()
    
    acc_history = []
    step_intervals = 500
    
    t0 = time.time()
    for i, (idx, row) in enumerate(X_train.head(10000).iterrows()):
        x_dict = row.to_dict()
        y_val = int(y_train[i])
        
        # Prequential Evaluation: test-then-train
        if i > 50:
            y_pred_stream = ht_model.predict_one(x_dict)
            if y_pred_stream is not None:
                stream_acc.update(y_val, y_pred_stream)
                
        ht_model.learn_one(x_dict, y_val)
        
        if i % step_intervals == 0 and i > 0:
            acc_history.append((i, stream_acc.get()))
            
    stream_time = time.time() - t0
    print(f"Streamed 10,000 transactions in {stream_time:.2f}s ({10000/stream_time:.0f} rows/sec)")
    print(f"Final Online Accuracy: {stream_acc.get():.4f}")
    
    steps, accs = zip(*acc_history)
    plt.figure(figsize=(10, 4))
    plt.plot(steps, accs, marker='o', color='forestgreen', linewidth=2)
    plt.title('Hoeffding Tree Real-Time Online Learning Trajectory', fontsize=12, fontweight='bold')
    plt.xlabel('Streaming Transaction Number', fontsize=11)
    plt.ylabel('Prequential Accuracy', fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("river not installed; skipping streaming simulation.")

### 5. Model 3: Unsupervised Fraud & Outlier Isolation (`IsolationForest`)
Isolating anomalous bot orders and suspicious credit transactions via average random tree path length.

In [ ]:
from sklearn.ensemble import IsolationForest

iso_forest = IsolationForest(n_estimators=100, contamination=0.03, random_state=42, n_jobs=-1)
iso_forest.fit(X_train[num_cols])

anomaly_scores = iso_forest.decision_function(X_test[num_cols])
is_outlier = iso_forest.predict(X_test[num_cols]) == -1

print(f"Total Test Transactions : {len(X_test):,}")
print(f"Flagged Outlier Orders  : {np.sum(is_outlier):,} ({np.mean(is_outlier):.1%})")

plt.figure(figsize=(9, 4))
plt.hist(anomaly_scores, bins=50, color='crimson', edgecolor='black', alpha=0.75)
plt.axvline(x=0.0, color='black', linestyle='--', label='Anomaly Threshold')
plt.title('Isolation Forest Path Length Anomaly Distribution', fontsize=12, fontweight='bold')
plt.xlabel('Isolation Path Score (s < 0 = Anomaly)', fontsize=11)
plt.ylabel('Transaction Count', fontsize=11)
plt.legend()
plt.tight_layout()
plt.show()

### 6. Summary: Production Decision Guide

| Production Constraint | Recommended Architecture | Mathematical Justification |
|---|---|---|
| **Sub-Millisecond Serving (<10 μs)** | **Google YDF** | Flat contiguous memory layout eliminates pointer-chasing. |
| **Continuous Streaming (Kafka)** | **Hoeffding Tree (`river`)** | Hoeffding Bound guarantees split optimality on $O(1)$ single passes. |
| **Calibrated Prediction Intervals**| **NGBoost** | Natural Gradient updates on Riemannian Fisher information manifold. |
| **Zero-Label Outlier Detection** | **Isolation Forest** | Average BST search length $c(n) = 2\ln(n-1) + 2\gamma - \frac{2(n-1)}{n}$. |